In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, KFold, cross_validate, cross_val_score
from sklearn import metrics

import sys
sys.path.insert(0, '../..')

from sn_clf.scripts.utils import get_sn_label_from_akb, load_features, download_akb_json

In [5]:
bts_sn = pd.read_csv('../data/bts_crossmatched_2sec.csv')

In [6]:
#Crossmatch ojects from BTS and OIDs file
oids, features = load_features('../../dr23-features/sid_snad_clf_r_100.dat', '../../dr23-features/feature_snad_clf_r_100.dat')

t = time.monotonic()
bts_oids = list(bts_sn['OID'])
crossmatch = np.isin(oids, bts_oids)
print(f'Crossmatched in {np.round((time.monotonic() - t) / 60)} min')
np.save(f'../data/bts_dr23_crossmatch.npy', crossmatch)

crossmatch = np.load(f'../data/bts_dr23_crossmatch.npy')

In [3]:
#download_akb_json('akb_objects.json')
akb_oids, akb_sn_label = get_sn_label_from_akb('../data/akb_objects.json')

In [7]:
akb_oid_sn = akb_oids[akb_sn_label == 1]
akb_crossmatch = np.isin(oids, akb_oid_sn)

In [9]:
np.save(f'../data/akb_sn_dr23_crossmatch.npy', akb_crossmatch)

In [8]:
model_name = 'SNclf_bts'
random_seed = 42
#Choose features, which will be used in training process
feature_names = '../../dr23-features/feature_snad_clf_r_100.name'
with open(feature_names) as f:
    names = f.read().split()

# тут в качестве негативного класса используются рандомные объекты из дата релиза
bts_features = features[crossmatch] # содержит только SN
akb_sn_features = features[akb_crossmatch]

indx = np.random.choice(np.arange(len(oids)), len(bts_features)+len(akb_sn_features))
regular_obj = features[indx]
train_data = np.vstack([bts_features, regular_obj[:len(bts_features)]])
test_data = np.vstack([akb_sn_features, regular_obj[len(bts_features):]])

train_labels = np.hstack([np.ones(len(bts_features)), np.zeros(len(bts_features))])
test_labels = np.hstack([np.ones(len(akb_sn_features)), np.zeros(len(akb_sn_features))])

In [10]:
# Train and validate real-bogus model
print('Training model...')
t = time.monotonic()
model = RandomForestClassifier(max_depth=18, n_estimators=831, random_state=random_seed)
score_types = ('accuracy', 'roc_auc', 'f1')

result = cross_validate(model, train_data, train_labels,
                        cv=KFold(shuffle=True, random_state=random_seed),
                        scoring=score_types,
                        return_estimator=True,
                        return_train_score=True,
                        n_jobs=5
                       )

print('Scores for Random Forest Classifier:')
for score in score_types:
    mean = np.mean(result[f'test_{score}'])
    std = np.std(result[f'test_{score}'])
    print(f'{score} = {mean:.3f} +- {std:.3f}')
t = (time.monotonic() - t) / 60
print(f'RF trained (with cross-validation) in {t:.0f} m')
    
#assert np.mean(result['test_accuracy']) > 0.7, 'Accuracy for trained model is too low!'
clf = result['estimator'][0]

#convert_to_onnx(clf, len(names), name=model_name)

Training model...
Scores for Random Forest Classifier:
accuracy = 0.952 +- 0.006
roc_auc = 0.987 +- 0.003
f1 = 0.952 +- 0.005
RF trained (with cross-validation) in 2 m


In [14]:
test_pred = clf.predict_proba(test_data)
akb_roc_auc = metrics.roc_auc_score(test_labels, test_pred[:, 1])
akb_accuracy = clf.score(test_data, test_labels)
print('Results on test data (SN from akb):')
print(f'ROC-AUC = {akb_roc_auc:.3f}')
print(f'Accuracy = {akb_accuracy:.3f}')

Results on test data (SN from akb):
ROC-AUC = 0.982
Accuracy = 0.909


0.9092827004219409